---
title: "Workshop for day 2"
author: "Inspired by Rasmus F Brøndum"
date: "23/05/2025"
format:
  html:
    toc: true
    toc-float: true
---


This document provides the basis for loading the data and training some dynamic classification models using Python libraries like `scikit-learn`, `xgboost`, and `optuna`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, precision_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
import multiprocessing

np.random.seed(2025)
cores = multiprocessing.cpu_count()

Load and read the data.

In [ ]:
df = pd.read_csv("Data_ready_for_workshop2.csv")

df

In [ ]:
df_cal = df[df['split'] == 'calibration'].copy()
df = df[df['split'] != 'calibration'].copy()

df['split'] = pd.Categorical(df['split'], categories=['train', 'validation', 'test'], ordered=True)

df.sort_values('split', inplace=True)

split_counts = df['split'].value_counts()

# Prepare data
def prepare_data(data):
    X = data.drop(columns=['id', 'date', 'split', 'mevent_nyear'])
    y = LabelEncoder().fit_transform(data['mevent_nyear'])  # ensure 1 is positive class
    return X, y

X, y = prepare_data(df)
X_cal, y_cal = prepare_data(df_cal)
X.reset_index(drop = True)
X_cal.reset_index(drop = True)

Data splitting

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=split_counts['test'], shuffle = False)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=split_counts['validation'], shuffle = False)

Basic Logistic Regression

In [ ]:
blr_model = LogisticRegression(max_iter=1000)
blr_model.fit(X_train, y_train)
y_test_pred = blr_model.predict(X_test)
y_test_prob = blr_model.predict_proba(X_test)[:, 1]

print("Accuracy:", (y_test_pred == y_test).mean())
print("ROC AUC:", roc_auc_score(y_test, y_test_prob))

ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_test_prob)
plt.plot(fpr, tpr, label='Logistic Regression')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid()
plt.axline((0, 0), (1, 1), linewidth=1, linestyle = '--', color='r')
plt.show()
plt.clf()

Elastic Net Logistic Regression (Grid Search)

In [ ]:
from sklearn.model_selection import GridSearchCV, PredefinedSplit

param_grid = {
    'C': 1 / np.logspace(-4, 0, 101),  # Inverse of regularization
    'l1_ratio': np.linspace(0, 1, 11)
}

# Using `saga` solver for elastic net logistic regression
lr_model = LogisticRegression(penalty='elasticnet', solver='saga', max_iter=5000)

test_fold = np.array([-1]*len(X_train) + [0]*len(X_val))

split = PredefinedSplit(test_fold)

lr_grid = GridSearchCV(
    lr_model,
    param_grid = {
        'C': 1 / np.logspace(-4, 0, 5),
        'l1_ratio': np.linspace(0, 1, 5)
    },
    scoring = 'roc_auc', cv = split, verbose = 2, n_jobs = cores
)

lr_grid.fit(X_train_val, y_train_val)
print("Best parameters:", lr_grid.best_params_)

ROC Curve

In [ ]:
lr_best = lr_grid.best_estimator_
y_test_prob = lr_best.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_test_prob)
plt.plot(fpr, tpr, label='Elastic Net Logistic Regression')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.legend()
plt.axline((0, 0), (1, 1), linewidth=1, linestyle = '--', color='r')
plt.grid()
plt.show()
plt.clf()

Random Forest

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

rf_model = RandomForestClassifier(n_estimators=100, n_jobs=cores)

rf_grid = {
    'max_features': ['sqrt', 'log2', None],
    'min_samples_leaf': [1, 2, 4, 6, 8]
}

rf_search = RandomizedSearchCV(
    rf_model, rf_grid, n_iter=15, cv=split, scoring='roc_auc', n_jobs=cores
)
rf_search.fit(X_train_val, y_train_val)
print("Best RF parameters:", rf_search.best_params_)

ROC

In [ ]:
rf_best = rf_search.best_estimator_
y_test_prob = rf_best.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_test_prob)
plt.plot(fpr, tpr, label='Random Forest')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.legend()
plt.axline((0, 0), (1, 1), linewidth=1, linestyle = '--', color='r')
plt.grid()
plt.show()
plt.clf()

Gradient Boosted Trees (XGBoost)

In [ ]:
gb_model = XGBClassifier(n_estimators=250, n_jobs=cores, eval_metric='logloss')

gb_grid = {
    'min_child_weight': [1, 5, 10],
    'gamma': [0, 0.1, 0.2],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.1]
}

gb_search = RandomizedSearchCV(
    gb_model, gb_grid, n_iter=25, scoring='roc_auc', n_jobs=cores, cv=split
)
gb_search.fit(X_train_val, y_train_val)
print("Best GB parameters:", gb_search.best_params_)

ROC

In [ ]:
gb_best = gb_search.best_estimator_
y_test_prob = xgb_best.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_test_prob)
plt.plot(fpr, tpr, label='XGBoost')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.legend()
plt.axline((0, 0), (1, 1), linewidth=1, linestyle = '--', color='r')
plt.grid()
plt.show()
plt.clf()

Multi-layer Perceptron (MLP)

In [ ]:
mlp_grid = {
    'hidden_layer_sizes': [(10,), (30,), (50,)],
    'alpha': [0.00001, 0.0001, 0.001, 0.01, 0.1],
    'learning_rate_init': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
}

mlp_model = MLPClassifier(max_iter=1000, activation='logistic')

mlp_search = GridSearchCV(mlp_model, mlp_grid, scoring='roc_auc', cv=split, n_jobs=cores)
mlp_search.fit(X_train_val, y_train_val)
print("Best MLP parameters:", mlp_search.best_params_)

ROC

In [ ]:
mlp_best = mlp_search.best_estimator_
y_val_prob = mlp_best.predict_proba(X_val)[:, 1]
fpr, tpr, _ = roc_curve(y_val, y_val_prob)
plt.plot(fpr, tpr, label='MLP')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.legend()
plt.axline((0, 0), (1, 1), linewidth=1, linestyle = '--', color='r')
plt.grid()
plt.show()
plt.clf()

Final Evaluation on Test Set

In [ ]:
models = {
    'Logistic Regression': blr_model,
    'Elastic Net': lr_best,
    'Random Forest': rf_best,
    'XGBoost': gb_best,
    'MLP': mlp_best
}

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    print(f"{name}")
    print("Precision:", precision_score(y_test, y_pred, zero_division = 0))
    print("ROC AUC:", roc_auc_score(y_test, y_prob))
    print("PR AUC:", average_precision_score(y_test, y_prob))
    print()

Feature Importance (when applicable)

In [ ]:
from sklearn.inspection import permutation_importance
from itertools import chain
import shap
from sklearn.datasets import load_iris

for name, model in models.items():
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    elif name in {'Elastic Net', 'Logistic Regression'}:
        importances = np.array(list(chain.from_iterable([abs(number) for number in model.coef_])))
    else:
        # Use KernelExplainer (model-agnostic)
        explainer = shap.KernelExplainer(model.predict_proba, X_train[:50], rng = 42)
        shap_values = explainer.shap_values(X_test[:10], rng = 42)  # smaller subset for speed
        shap_array = np.array(shap_values[1])
        importances = np.abs(shap_array).mean(axis=0)

    top_idx = np.argsort(importances)[::-1][:20]
    plt.figure(figsize=(5,3))
    plt.barh(np.array(X.columns)[top_idx], importances[top_idx])
    plt.title(f"Top Features - {name}")
    plt.tight_layout()
    plt.gca().invert_yaxis()
    plt.show()
    plt.clf()

Calibration model.